## Se hace comparativa con diferentes Hiperparámetros

### Usaremos RAG + DSLR


In [ ]:
### Cargamos el modelo phi-4
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id = "microsoft/phi-4-mini-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id,padding_side="left")
model_lm = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", torch_dtype="auto")
generator = pipeline("text-generation", model=model_lm, tokenizer=tokenizer,batch_size=8,device_map="auto",dtype="auto")

Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.01it/s]
Some parameters are on the meta device because they were offloaded to the cpu.
Device set to use cuda:0


In [29]:
import pickle
#Usaremos los embedding y metadatos usados anteriormente
with open(f"../outputs/embeddings_gemma_y_metadatos.pkl", "rb") as f:
    data = pickle.load(f)

embeddings = data["embeddings"]
metadatos = data["metadatos"]

In [30]:
#Uso FAISS para indexar
#FAISS es una librería desarrollada por Meta (Facebook) para hacer búsqueda rápida de vectores por similitud, ideal cuando tienes muchos embeddings (como en RAG).
import faiss
import numpy as np
embedding_matrix = np.array(embeddings)

# Crear índice FAISS (búsqueda por similitud L2 o Euclidiana)
dim = embedding_matrix.shape[1] 
print(dim)
index = faiss.IndexFlatL2(dim)  

# Agregar los vectores al índice
index.add(embedding_matrix)

# Guardar el índice en disco
faiss.write_index(index, "../outputs/faiss_index.index")

768


In [31]:
import pandas as pd
# Cargar el archivo CSV
df_qa = pd.read_csv("../dataQA/qa.txt")
df_qa.head()

,chunk,question,answer
0,seawater seawater tds mgl considered standard ...,What types of water are classified based on To...,Water is classified into categories like seawa...
1,recovery limit salinity andor boron concentrat...,Why is it important to limit product recovery ...,Limiting product recovery is important to ensu...
2,design boynton beach fl membrane softening wat...,How is the maximum recovery value determined f...,The maximum recovery value is determined by co...
3,range rather absolute value temperature variat...,Why is average temperature used for performanc...,Average temperature is used because membrane p...
4,risk scaling due water scarcity environmental ...,Why must scaling substances be removed from tr...,"Even after secondary treatment, wastewater can..."


### DSLR

In [ ]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

In [32]:
import spacy
nlp = spacy.load("en_core_web_sm")

def dividir_oraciones(texto):
    doc = nlp(texto)
    return [sent.text.strip() for sent in doc.sents]

In [ ]:
from sentence_transformers import CrossEncoder


def re_rank_oraciones(question, oraciones):
    pairs = [(question, sent) for sent in oraciones]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(oraciones, scores), key=lambda x: x[1], reverse=True)
    return ranked

In [34]:
import numpy as np

def filtrar_por_umbral(oraciones_ranked, percentil=90):
    scores = [score for _, score in oraciones_ranked]
    umbral = np.percentile(scores, percentil)
    oraciones_filtradas = [(sent, score) for sent, score in oraciones_ranked if score >= umbral]
    return oraciones_filtradas, umbral

In [35]:
def reconstruir_contexto(oraciones_filtradas, oraciones_originales):
    oraciones_validas = set([sent for sent, _ in oraciones_filtradas])
    reconstruido = [sent for sent in oraciones_originales if sent in oraciones_validas]
    return reconstruido

In [36]:
## Funcion para implementar los 3 pasos de DSLR 
def refine_documents(sentence,pregunta,treshhold=90):
    # Paso 1: Separar oraciones
    oraciones = dividir_oraciones(sentence)

    # Paso 2: Rankear oraciones
    oraciones_ranked = re_rank_oraciones(pregunta, oraciones)

    # Paso 3: Filtrar con percentil 90
    oraciones_filtradas, umbral = filtrar_por_umbral(oraciones_ranked, percentil=treshhold)

    # Paso 4: Reconstruir en orden original
    oraciones_reconstruidas = reconstruir_contexto(oraciones_filtradas, oraciones)

    # Resultado final para pasar al LLM
    documento_refinado = " ".join(oraciones_reconstruidas)
    return documento_refinado
         

In [37]:
import torch
## Implementamos DSLR en nuestro pipeline
def responder_con_phi4_con_contexto_refinado(
        preguntas, 
        modelo_embedding,
        treshhold=90,
        temperature=0.5,
        k=5,
        inEnglish=False
    ):
    
    resultados = []
    idioma = "The answer must be in English." if inEnglish else ""

    # Embeddings del batch completo (esto YA está en batch)
    preguntas_vec = modelo_embedding.encode(preguntas, convert_to_tensor=True)

    # Recuperación con FAISS en batch
    D, I = index.search(preguntas_vec.cpu().numpy(), k)

    # Preparamos prompts y metadatos para el batch
    prompts = []
    chunks_info = []

    for idx_preg, pregunta in enumerate(preguntas):
        contexto = ""
        used_chunks = []

        for idx in I[idx_preg]:
            doc = metadatos[idx]
            chunk_text = doc["chunk"].strip()
            titulo = doc.get("id_doc", "Sin título")

            texto_refinado = refine_documents(chunk_text, pregunta, treshhold)
            used_chunks.append({"titulo": titulo, "chunk": texto_refinado})
            contexto += f"- {texto_refinado}\n"

        prompt = (
            f"<|user|>\nUsa el siguiente contexto para responder la pregunta:\n\n"
            f"Contexto:\n{contexto}\nPregunta: {pregunta} {idioma}\n<|assistant|>"
        )

        prompts.append(prompt)
        chunks_info.append(used_chunks)


    with torch.no_grad():
        outputs = generator(
            prompts,
            max_new_tokens=300,
            temperature=temperature,
            do_sample=True,
        )

    # Procesamos respuestas
    for i, out in enumerate(outputs): 
        # Cortamos el prompt
        full_text = out[0]["generated_text"]
        respuesta = full_text[len(prompts[i]):].strip()

        resultados.append({
            "pregunta": preguntas[i],
            "respuesta": respuesta,
            "chunks_usados": chunks_info[i]
        })

    return resultados


In [38]:
from sentence_transformers import SentenceTransformer
import itertools
import torch

temperatures = [0.1, 0.3]
top_k_values = [3, 5, 8]
treshholds = [90, 95]
batch_size = 32

preguntas = df_qa["question"].tolist()
modelo_embedding = SentenceTransformer("google/embeddinggemma-300m")

# Todas las combinaciones
param_grid = list(itertools.product(temperatures, top_k_values, treshholds))

for temp, top_k, tresh in param_grid:
    print(f"\n=== Procesando con temperature={temp}, top_k={top_k}, tresh={tresh} ===")

    for start in range(0, len(preguntas), batch_size):
        batch = preguntas[start:start + batch_size]

        resultados_batch = responder_con_phi4_con_contexto_refinado(
            batch,
            modelo_embedding,
            treshhold=tresh,
            temperature=temp,
            k=top_k,
            inEnglish=True
        )

        # Guardar resultados (incluyendo los 3 hiperparámetros)
        for i, resultado in enumerate(resultados_batch):
            idx = start + i
            df_qa.at[idx, f"ans_temp_{temp}_k_{top_k}_tresh_{tresh}"] = resultado["respuesta"]

    torch.cuda.empty_cache()



=== Procesando con temperature=0.1, top_k=3, tresh=90 ===

=== Procesando con temperature=0.1, top_k=3, tresh=95 ===

=== Procesando con temperature=0.1, top_k=5, tresh=90 ===


HTTP Error 504 thrown while requesting HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/README.md
Retrying in 1s [Retry 1/5].
HTTP Error 504 thrown while requesting HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/README.md
Retrying in 2s [Retry 2/5].
HTTP Error 504 thrown while requesting HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
HTTP Error 504 thrown while requesting HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/tokenizer_config.json
Retrying in 2s [Retry 2/5].



=== Procesando con temperature=0.1, top_k=5, tresh=95 ===

=== Procesando con temperature=0.1, top_k=8, tresh=90 ===

=== Procesando con temperature=0.1, top_k=8, tresh=95 ===

=== Procesando con temperature=0.3, top_k=3, tresh=90 ===

=== Procesando con temperature=0.3, top_k=3, tresh=95 ===


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 799a1ca4-2e4c-46d8-8dbc-a0a346eef523)')' thrown while requesting HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].



=== Procesando con temperature=0.3, top_k=5, tresh=90 ===

=== Procesando con temperature=0.3, top_k=5, tresh=95 ===

=== Procesando con temperature=0.3, top_k=8, tresh=90 ===


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 34d5a58d-7023-4d20-be3a-b93f662fed79)')' thrown while requesting HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].



=== Procesando con temperature=0.3, top_k=8, tresh=95 ===


In [47]:
df_qa.head()

,chunk,question,answer,ans_temp_0.1_k_3_tresh_90,ans_temp_0.1_k_3_tresh_95,ans_temp_0.1_k_5_tresh_90,ans_temp_0.1_k_5_tresh_95,ans_temp_0.1_k_8_tresh_90,ans_temp_0.1_k_8_tresh_95,ans_temp_0.3_k_3_tresh_90,ans_temp_0.3_k_3_tresh_95,ans_temp_0.3_k_5_tresh_90,ans_temp_0.3_k_5_tresh_95,ans_temp_0.3_k_8_tresh_90,ans_temp_0.3_k_8_tresh_95
0,seawater seawater tds mgl considered standard ...,What types of water are classified based on To...,Water is classified into categories like seawa...,"Based on the provided context, water is classi...","Based on the provided context, water can be cl...","Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ...",Water is classified based on Total Dissolved S...,"Based on the provided context, water can be cl...","Based on Total Dissolved Solids (TDS) levels, ...",Water can be classified based on Total Dissolv...,"Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ..."
1,recovery limit salinity andor boron concentrat...,Why is it important to limit product recovery ...,Limiting product recovery is important to ensu...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...
2,design boynton beach fl membrane softening wat...,How is the maximum recovery value determined f...,The maximum recovery value is determined by co...,The provided context does not explicitly state...,The provided context does not explicitly state...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...,The provided context does not explicitly state...,The provided context does not explicitly state...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...
3,range rather absolute value temperature variat...,Why is average temperature used for performanc...,Average temperature is used because membrane p...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...
4,risk scaling due water scarcity environmental ...,Why must scaling substances be removed from tr...,"Even after secondary treatment, wastewater can...",Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be remove

In [40]:
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [41]:
df_qa.describe()

,chunk,question,answer,ans_temp_0.1_k_3_tresh_90,ans_temp_0.1_k_3_tresh_95,ans_temp_0.1_k_5_tresh_90,ans_temp_0.1_k_5_tresh_95,ans_temp_0.1_k_8_tresh_90,ans_temp_0.1_k_8_tresh_95,ans_temp_0.3_k_3_tresh_90,ans_temp_0.3_k_3_tresh_95,ans_temp_0.3_k_5_tresh_90,ans_temp_0.3_k_5_tresh_95,ans_temp_0.3_k_8_tresh_90,ans_temp_0.3_k_8_tresh_95
count,511,511,511,511,511,511,511,511,511,511,511,511,511,511,511
unique,511,511,511,511,511,511,511,511,511,511,511,511,511,511,511
top,seawater seawater tds mgl considered standard ...,What types of water are classified based on To...,Water is classified into categories like seawa...,"Based on the provided context, water is classi...","Based on the provided context, water can be cl...","Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ...",Water is classified based on Total Dissolved S...,"Based on the provided context, water can be cl...","Based on Total Dissolved Solids (TDS) levels, ...",Water can be classified based on Total Dissolv...,"Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ..."
freq,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


In [3]:
df_qa.head()

,Unnamed: 0,chunk,question,answer,ans_temp_0.1_k_3_tresh_90,ans_temp_0.1_k_3_tresh_95,ans_temp_0.1_k_5_tresh_90,ans_temp_0.1_k_5_tresh_95,ans_temp_0.1_k_8_tresh_90,ans_temp_0.1_k_8_tresh_95,ans_temp_0.3_k_3_tresh_90,ans_temp_0.3_k_3_tresh_95,ans_temp_0.3_k_5_tresh_90,ans_temp_0.3_k_5_tresh_95,ans_temp_0.3_k_8_tresh_90,ans_temp_0.3_k_8_tresh_95
0,0,seawater seawater tds mgl considered standard ...,What types of water are classified based on To...,Water is classified into categories like seawa...,"Based on the provided context, water is classi...","Based on the provided context, water can be cl...","Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ...",Water is classified based on Total Dissolved S...,"Based on the provided context, water can be cl...","Based on Total Dissolved Solids (TDS) levels, ...",Water can be classified based on Total Dissolv...,"Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ..."
1,1,recovery limit salinity andor boron concentrat...,Why is it important to limit product recovery ...,Limiting product recovery is important to ensu...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...
2,2,design boynton beach fl membrane softening wat...,How is the maximum recovery value determined f...,The maximum recovery value is determined by co...,The provided context does not explicitly state...,The provided context does not explicitly state...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...,The provided context does not explicitly state...,The provided context does not explicitly state...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...
3,3,range rather absolute value temperature variat...,Why is average temperature used for performanc...,Average temperature is used because membrane p...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...
4,4,risk scaling due water scarcity environmental ...,Why must scaling substances be removed from tr...,"Even after secondary treatment, wastewater can...",Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling subs

In [43]:
df_qa.head(100).describe()

,chunk,question,answer,ans_temp_0.1_k_3_tresh_90,ans_temp_0.1_k_3_tresh_95,ans_temp_0.1_k_5_tresh_90,ans_temp_0.1_k_5_tresh_95,ans_temp_0.1_k_8_tresh_90,ans_temp_0.1_k_8_tresh_95,ans_temp_0.3_k_3_tresh_90,ans_temp_0.3_k_3_tresh_95,ans_temp_0.3_k_5_tresh_90,ans_temp_0.3_k_5_tresh_95,ans_temp_0.3_k_8_tresh_90,ans_temp_0.3_k_8_tresh_95
count,100,100,100,100,100,100,100,100,100,100,100,100,100,100,100
unique,100,100,100,100,100,100,100,100,100,100,100,100,100,100,100
top,seawater seawater tds mgl considered standard ...,What types of water are classified based on To...,Water is classified into categories like seawa...,"Based on the provided context, water is classi...","Based on the provided context, water can be cl...","Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ...",Water is classified based on Total Dissolved S...,"Based on the provided context, water can be cl...","Based on Total Dissolved Solids (TDS) levels, ...",Water can be classified based on Total Dissolv...,"Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ..."
freq,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1


In [48]:
df_qa.to_csv("../dataQA/hiperparameters_grid.csv")

In [55]:
df_qa["ans_temp_0.1_k_3_tresh_95"][1]

"Limiting product recovery in RO (Reverse Osmosis) systems is important for several reasons, particularly in the context of brackish water treatment where chemical factors like precipitation and scale formation are significant. Firstly, excessive product recovery can lead to increased concentration of dissolved solids in the concentrate stream, which can cause scaling and fouling of the RO membranes. Scaling, primarily caused by compounds such as calcium carbonate or calcium sulfate, can reduce the efficiency of the RO system, increase energy consumption, and ultimately lead to system downtime for cleaning or membrane replacement.\n\nSecondly, maintaining a controlled product recovery rate helps in managing the chemical balance within the system. In brackish water treatment, the chemical nature of the water can lead to precipitation and scaling if not properly managed. By limiting product recovery, the concentration of these scaling compounds in the concentrate stream is reduced, there

In [56]:
df_qa["ans_temp_0.1_k_5_tresh_95"][1]

"Limiting product recovery in RO (Reverse Osmosis) systems is important to minimize precipitation and scaling. When the solubility limits of sparingly soluble salts, such as calcium carbonate or calcium sulfate, are exceeded, it can lead to the formation of scale on the membranes. This scale can reduce the efficiency of the RO system, increase energy consumption, and potentially cause damage to the membranes, leading to costly repairs or replacements. Additionally, maintaining the system within the solubility limits helps ensure the longevity and optimal performance of the RO system. Proper scale control measures are essential to avoid these issues and maintain the system's effectiveness in water treatment processes."

In [57]:
df_qa["ans_temp_0.3_k_3_tresh_90"][1]

"Limiting product recovery in RO (Reverse Osmosis) systems is important for several reasons:\n\n1. **Prevention of Scale Formation**: In brackish water treatment, the recovery of water can lead to increased concentrations of dissolved salts, such as calcium carbonate or calcium sulfate. These salts can precipitate and form scale on the RO membranes, which can reduce the efficiency of the system and lead to increased maintenance costs.\n\n2. **Membrane Longevity**: High recovery rates can cause more frequent scaling and fouling of the RO membranes. This can shorten the lifespan of the membranes and lead to more frequent replacements, increasing operational costs.\n\n3. **System Efficiency**: Operating the RO system above the solubility product (Ksp) value without adequate scale inhibitors can lead to precipitation and scaling, which can clog the system and reduce its overall efficiency. By limiting recovery, the concentration of potential scaling compounds is kept lower, maintaining the

### ROUGUE SCORE

In [5]:
import os
output_dir = os.path.join("..", "resultados")
# El ROUGE score (Recall-Oriented Understudy for Gisting Evaluation) es una métrica ampliamente usada para evaluar la calidad de textos generados automáticamente
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
## Definimos funcion para hallar ROUGE
def calculate_rouge_score(column_name,data_name):
    # Evaluar ROUGE para cada par respuesta_modelo - respuesta_referencia
    rouge_scores = []
    
    for i, row in df_qa.iterrows():
        ref = row["answer"]  # respuesta de referencia
        gen = row[column_name]  # respuesta generada
        score = scorer.score(ref, gen)
        rouge_scores.append({
            "ROUGE-1": score["rouge1"].fmeasure,
            "ROUGE-2": score["rouge2"].fmeasure,
            "ROUGE-L": score["rougeL"].fmeasure
        })
        
    #Resultados
    # Convertir a DataFrame y unirlo al original
    df_rouge = pd.DataFrame(rouge_scores)
    df_resultado = pd.concat([df_qa.reset_index(drop=True), df_rouge], axis=1)

    # Mostrar puntajes promedio
    promedios = df_rouge.mean()
    print("🔍 Promedios ROUGE:")
    print(promedios.round(4))
    
    # Ver resultados por pregunta
    print("\n📌 Ejemplos con ROUGE:")
    print(df_resultado[["question", "ROUGE-1", "ROUGE-2", "ROUGE-L"]])

    # Guardar
    output_path = os.path.join(output_dir, data_name)
    df_resultado.to_csv(output_path, index=False)
    print(f"\n✅ Guardado en {data_name}")

In [ ]:
cols_ans = df_qa.filter(regex="^ans_").columns
for col in cols_ans: 
    file_name= col.split(".", 1)[1]
    calculate_rouge_score(col,f"busqueda_hiperparametros/evaluacion_dslr_con_rouge_h_{file_name}.csv")

🔍 Promedios ROUGE:
ROUGE-1    0.2358
ROUGE-2    0.0750
ROUGE-L    0.1723
dtype: float64

📌 Ejemplos con ROUGE:
                                              question   ROUGE-1   ROUGE-2  \
0    What types of water are classified based on To...  0.176166  0.073298   
1    Why is it important to limit product recovery ...  0.155642  0.031373   
2    How is the maximum recovery value determined f...  0.244186  0.094118   
3    Why is average temperature used for performanc...  0.260870  0.073529   
4    Why must scaling substances be removed from tr...  0.258065  0.065574   
..                                                 ...       ...       ...   
506  What are common causes for failed serial commu...  0.095941  0.029740   
507  How should you troubleshoot incorrect output b...  0.177606  0.054475   
508  How can you fix alarm and control mode issues ...  0.178862  0.081967   
509  What steps can correct poor control accuracy o...  0.091255  0.000000   
510  How should error messages 

### BERT SCORE

In [13]:
from bert_score import score
import pandas as pd
import os

def calculate_bert_score(name_column, name_data):
    refs = df_qa["answer"].astype(str).tolist()
    gens = df_qa[name_column].astype(str).tolist()

    # Calcular BERTScore para todo el batch
    P, R, F1 = score(gens, refs, lang="en", verbose=False)

    df_bert = pd.DataFrame({
        "PRECISION": P.tolist(),
        "RECALL": R.tolist(),
        "F1": F1.tolist()
    })

    df_resultado_bert = pd.concat([df_qa.head(100).reset_index(drop=True), df_bert], axis=1)

    # Mostrar promedio
    promedios = df_bert.mean()
    print("🔍 Promedios BERTSCORE:")
    print(promedios.round(4))

    print("\n📌 Ejemplos con BERT:")
    print(df_resultado_bert[["question", "PRECISION", "RECALL", "F1"]].head(10))

    # Guardar
    output_path = os.path.join(output_dir, name_data)
    df_resultado_bert.to_csv(output_path, index=False)

c:\Users\PC\Maestria\NLP\RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
for col in cols_ans: 
    file_name= col.split(".", 1)[1]
    calculate_bert_score(col,f"busqueda_hiperparametros/evaluacion_dslr_con_bert_h_{file_name}.csv")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🔍 Promedios BERTSCORE:
PRECISION    0.8439
RECALL       0.8757
F1           0.8592
dtype: float64

📌 Ejemplos con BERT:
                                            question  PRECISION    RECALL  \
0  What types of water are classified based on To...   0.799323  0.859949   
1  Why is it important to limit product recovery ...   0.821136  0.881129   
2  How is the maximum recovery value determined f...   0.848589  0.898304   
3  Why is average temperature used for performanc...   0.856901  0.905166   
4  Why must scaling substances be removed from tr...   0.861156  0.901512   
5  What causes scaling due to supersaturation in ...   0.846156  0.886274   
6   How is the scaling tendency of salts quantified?   0.810096  0.839633   
7  What factors influence the effectiveness of an...   0.803164  0.874568   
8  What are the benefits of phosphate-based and p...   0.831346  0.881409   
9  Why must antiscalants be handled carefully in ...   0.848336  0.859561   

         F1  
0  0.828528  
1  0

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🔍 Promedios BERTSCORE:
PRECISION    0.8455
RECALL       0.8732
F1           0.8589
dtype: float64

📌 Ejemplos con BERT:
                                            question  PRECISION    RECALL  \
0  What types of water are classified based on To...   0.805912  0.853013   
1  Why is it important to limit product recovery ...   0.828705  0.880485   
2  How is the maximum recovery value determined f...   0.851166  0.901410   
3  Why is average temperature used for performanc...   0.854703  0.909304   
4  Why must scaling substances be removed from tr...   0.874345  0.905367   
5  What causes scaling due to supersaturation in ...   0.850573  0.883937   
6   How is the scaling tendency of salts quantified?   0.812620  0.841322   
7  What factors influence the effectiveness of an...   0.795418  0.858606   
8  What are the benefits of phosphate-based and p...   0.837430  0.884737   
9  Why must antiscalants be handled carefully in ...   0.840553  0.861760   

         F1  
0  0.828794  
1  0

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

🔍 Promedios BERTSCORE:
PRECISION    0.8422
RECALL       0.8749
F1           0.8580
dtype: float64

📌 Ejemplos con BERT:
                                            question  PRECISION    RECALL  \
0  What types of water are classified based on To...   0.850100  0.889938   
1  Why is it important to limit product recovery ...   0.849700  0.889532   
2  How is the maximum recovery value determined f...   0.831615  0.909632   
3  Why is average temperature used for performanc...   0.861901  0.909202   
4  Why must scaling substances be removed from tr...   0.849281  0.897089   
5  What causes scaling due to supersaturation in ...   0.824803  0.894294   
6   How is the scaling tendency of salts quantified?   0.822873  0.834072   
7  What factors influence the effectiveness of an...   0.793775  0.857212   
8  What are the benefits of phosphate-based and p...   0.838591  0.887506   
9  Why must antiscalants be handled carefully in ...   0.825606  0.857063   

         F1  
0  0.869563  
1  0

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🔍 Promedios BERTSCORE:
PRECISION    0.8390
RECALL       0.8777
F1           0.8577
dtype: float64

📌 Ejemplos con BERT:
                                            question  PRECISION    RECALL  \
0  What types of water are classified based on To...   0.839061  0.887062   
1  Why is it important to limit product recovery ...   0.841283  0.889604   
2  How is the maximum recovery value determined f...   0.827478  0.908260   
3  Why is average temperature used for performanc...   0.856552  0.912044   
4  Why must scaling substances be removed from tr...   0.851298  0.899559   
5  What causes scaling due to supersaturation in ...   0.816917  0.888936   
6   How is the scaling tendency of salts quantified?   0.825194  0.844627   
7  What factors influence the effectiveness of an...   0.793102  0.859502   
8  What are the benefits of phosphate-based and p...   0.824296  0.883546   
9  Why must antiscalants be handled carefully in ...   0.837407  0.879589   

         F1  
0  0.862394  
1  0

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🔍 Promedios BERTSCORE:
PRECISION    0.8396
RECALL       0.8756
F1           0.8570
dtype: float64

📌 Ejemplos con BERT:
                                            question  PRECISION    RECALL  \
0  What types of water are classified based on To...   0.827713  0.879157   
1  Why is it important to limit product recovery ...   0.845193  0.883153   
2  How is the maximum recovery value determined f...   0.853598  0.909961   
3  Why is average temperature used for performanc...   0.839824  0.914885   
4  Why must scaling substances be removed from tr...   0.847288  0.901332   
5  What causes scaling due to supersaturation in ...   0.822924  0.877984   
6   How is the scaling tendency of salts quantified?   0.819249  0.836829   
7  What factors influence the effectiveness of an...   0.795996  0.853598   
8  What are the benefits of phosphate-based and p...   0.826565  0.877471   
9  Why must antiscalants be handled carefully in ...   0.831942  0.863492   

         F1  
0  0.852659  
1  0

In [ ]:
df_qa_gemma= pd.read_csv("../dataQA/df_qa_gemma.csv")
df_qa_gemma.head()

,chunk,question,answer,answer_modelo_rag,retrieved,answer_modelo,claims_phi4,answer_modelo_rag_dslr,retrieved_dslr
0,seawater seawater tds mgl considered standard ...,What types of water are classified based on To...,Water is classified into categories like seawa...,"Based on Total Dissolved Solids (TDS) levels, ...","[{""titulo"": ""7.5 RO FOULING substance (anaysis...",Total Dissolved Solids (TDS) levels are used t...,"['Brackish Water TDS levels are between 1,000 ...",Water can be classified based on Total Dissolv...,"[{""titulo"": ""7.5 RO FOULING substance (anaysis..."
1,recovery limit salinity andor boron concentrat...,Why is it important to limit product recovery ...,Limiting product recovery is important to ensu...,Limiting product recovery in reverse osmosis (...,"[{""titulo"": ""7.5 RO FOULING substance (anaysis...",Limiting product recovery in Reverse Osmosis (...,['Limiting product recovery in RO systems is i...,Limiting product recovery in RO (Reverse Osmos...,"[{""titulo"": ""7.5 RO FOULING substance (anaysis..."
2,design boynton beach fl membrane softening wat...,How is the maximum recovery value determined f...,The maximum recovery value is determined by co...,El valor máximo de recuperación para sistemas ...,"[{""titulo"": ""7.5 RO FOULING substance (anaysis...","Membrane softening systems, such as those used...",['El valor máximo de recuperación se determina...,La máxima recuperación permitida para sistemas...,"[{""titulo"": ""7.5 RO FOULING substance (anaysis..."
3,range rather absolute value temperature variat...,Why is average temperature used for performanc...,Average temperature is used because membrane p...,The average temperature is used for performanc...,"[{""titulo"": ""7.5 RO FOULING substance (anaysis...","The term ""RO membranes"" typically refers to re...",['The average temperature is used for performa...,The average temperature is used for performanc...,"[{""titulo"": ""7.5 RO FOULING substance (anaysis..."
4,risk scaling due water scarcity environmental ...,Why must scaling substances be removed from tr...,"Even after secondary treatment, wastewater can...",Scaling substances must be removed from treate...,"[{""titulo"": ""7.5 RO FOULING substance (anaysis...","In reverse osmosis (RO) systems, scaling subst...",['Scaling substances must be removed from trea...,Scaling substances must be removed from treate...,"[{""titulo"": ""7.5 RO FOULING substance (anaysis..."


In [ ]:
df_qa["answer_modelo_rag_dslr_h_0.5"] = df_qa_gemma["answer_modelo_rag_dslr"]

In [ ]:
calculate_rouge_score("answer_modelo_rag_dslr_h_0.5","evaluacion_dslr_con_rouge_h_5.csv")

🔍 Promedios ROUGE:
ROUGE-1    0.2058
ROUGE-2    0.0627
ROUGE-L    0.1573
dtype: float64

📌 Ejemplos con ROUGE:
                                              question   ROUGE-1   ROUGE-2  \
0    What types of water are classified based on To...  0.259542  0.062016   
1    Why is it important to limit product recovery ...  0.252252  0.128440   
2    How is the maximum recovery value determined f...  0.000000  0.000000   
3    Why is average temperature used for performanc...  0.272109  0.096552   
4    Why must scaling substances be removed from tr...  0.262295  0.050000   
..                                                 ...       ...       ...   
506  What are common causes for failed serial commu...       NaN       NaN   
507  How should you troubleshoot incorrect output b...       NaN       NaN   
508  How can you fix alarm and control mode issues ...       NaN       NaN   
509  What steps can correct poor control accuracy o...       NaN       NaN   
510  How should error messages 

In [ ]:
calculate_bert_score("answer_modelo_rag_dslr_h_0.5","evaluacion_dslr_con_bert_h_5.csv")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🔍 Promedios BERTSCORE:
PRECISION    0.8404
RECALL       0.8908
F1           0.8647
dtype: float64

📌 Ejemplos con BERT:
                                            question  PRECISION    RECALL  \
0  What types of water are classified based on To...   0.816443  0.869528   
1  Why is it important to limit product recovery ...   0.858346  0.889183   
2  How is the maximum recovery value determined f...   0.721848  0.850407   
3  Why is average temperature used for performanc...   0.847332  0.909696   
4  Why must scaling substances be removed from tr...   0.857442  0.900266   
5  What causes scaling due to supersaturation in ...   0.820067  0.891562   
6   How is the scaling tendency of salts quantified?   0.836627  0.840376   
7  What factors influence the effectiveness of an...   0.796512  0.853372   
8  What are the benefits of phosphate-based and p...   0.842581  0.890864   
9  Why must antiscalants be handled carefully in ...   0.829654  0.874948   

         F1  
0  0.842149  
1  0

### Comparando

In [ ]:
df_bert_h1 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_bert_h_1_k_3_tresh_90.csv")
df_bert_h2 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_bert_h_1_k_3_tresh_95.csv")
df_bert_h3 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_bert_h_1_k_5_tresh_90.csv")
df_bert_h4 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_bert_h_1_k_5_tresh_95.csv")
df_bert_h5 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_bert_h_1_k_8_tresh_90.csv")
df_bert_h6 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_bert_h_1_k_8_tresh_95.csv")
df_bert_h7 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_bert_h_3_k_3_tresh_90.csv")
df_bert_h8 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_bert_h_3_k_3_tresh_95.csv")
df_bert_h9 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_bert_h_3_k_5_tresh_90.csv")
df_bert_h10 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_bert_h_3_k_5_tresh_95.csv")
df_bert_h11 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_bert_h_3_k_8_tresh_90.csv")
df_bert_h12 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_bert_h_3_k_8_tresh_95.csv")
df_bert_h1.head()

,Unnamed: 0,chunk,question,answer,ans_temp_0.1_k_3_tresh_90,ans_temp_0.1_k_3_tresh_95,ans_temp_0.1_k_5_tresh_90,ans_temp_0.1_k_5_tresh_95,ans_temp_0.1_k_8_tresh_90,ans_temp_0.1_k_8_tresh_95,ans_temp_0.3_k_3_tresh_90,ans_temp_0.3_k_3_tresh_95,ans_temp_0.3_k_5_tresh_90,ans_temp_0.3_k_5_tresh_95,ans_temp_0.3_k_8_tresh_90,ans_temp_0.3_k_8_tresh_95,PRECISION,RECALL,F1
0,0.0,seawater seawater tds mgl considered standard ...,What types of water are classified based on To...,Water is classified into categories like seawa...,"Based on the provided context, water is classi...","Based on the provided context, water can be cl...","Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ...",Water is classified based on Total Dissolved S...,"Based on the provided context, water can be cl...","Based on Total Dissolved Solids (TDS) levels, ...",Water can be classified based on Total Dissolv...,"Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ...",0.799323,0.859949,0.828528
1,1.0,recovery limit salinity andor boron concentrat...,Why is it important to limit product recovery ...,Limiting product recovery is important to ensu...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,0.821136,0.881129,0.850076
2,2.0,design boynton beach fl membrane softening wat...,How is the maximum recovery value determined f...,The maximum recovery value is determined by co...,The provided context does not explicitly state...,The provided context does not explicitly state...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...,The provided context does not explicitly state...,The provided context does not explicitly state...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...,0.848589,0.898304,0.872739
3,3.0,range rather absolute value temperature variat...,Why is average temperature used for performanc...,Average temperature is used because membrane p...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,0.856901,0.905166,0.880372
4,4.0,risk scaling due water scarcity environmental ...,Why must scaling substances be removed from tr...,"Even after secondary treatment, wastewater can...",Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must 

In [20]:
mean_BERT_1=[]
mean_BERT_2=[]
mean_BERT_3=[]
mean_BERT_4=[]
mean_BERT_5=[]
mean_BERT_6=[]
mean_BERT_7=[]
mean_BERT_8=[]
mean_BERT_9=[]
mean_BERT_10=[]
mean_BERT_11=[]
mean_BERT_12=[]

mean_BERT_1.append(df_bert_h1["F1"].mean().round(4))
mean_BERT_2.append(df_bert_h2["F1"].mean().round(4))
mean_BERT_3.append(df_bert_h3["F1"].mean().round(4))
mean_BERT_4.append(df_bert_h4["F1"].mean().round(4))
mean_BERT_5.append(df_bert_h5["F1"].mean().round(4))
mean_BERT_6.append(df_bert_h6["F1"].mean().round(4))
mean_BERT_7.append(df_bert_h7["F1"].mean().round(4))
mean_BERT_8.append(df_bert_h8["F1"].mean().round(4))
mean_BERT_9.append(df_bert_h9["F1"].mean().round(4))
mean_BERT_10.append(df_bert_h10["F1"].mean().round(4))
mean_BERT_11.append(df_bert_h11["F1"].mean().round(4))
mean_BERT_12.append(df_bert_h12["F1"].mean().round(4))

df_mean = pd.DataFrame({

    "BERT-H-1-K-3-T-90": mean_BERT_1,
    "BERT-H-1-K-3-T-95": mean_BERT_2,
    "BERT-H-1-K-5-T-90": mean_BERT_3,
    "BERT-H-1-K-5-T-95": mean_BERT_4,
    "BERT-H-1-K-8-T-90": mean_BERT_5,
    "BERT-H-1-K-8-T-95": mean_BERT_6,
    "BERT-H-3-K-3-T-90": mean_BERT_7,
    "BERT-H-3-K-3-T-95": mean_BERT_8,
    "BERT-H-3-K-5-T-90": mean_BERT_9,
    "BERT-H-3-K-5-T-95": mean_BERT_10,
    "BERT-H-3-K-8-T-90": mean_BERT_11,
    "BERT-H-3-K-8-T-95": mean_BERT_12,
})

In [21]:
df_mean

,BERT-H-1-K-3-T-90,BERT-H-1-K-3-T-95,BERT-H-1-K-5-T-90,BERT-H-1-K-5-T-95,BERT-H-1-K-8-T-90,BERT-H-1-K-8-T-95,BERT-H-3-K-3-T-90,BERT-H-3-K-3-T-95,BERT-H-3-K-5-T-90,BERT-H-3-K-5-T-95,BERT-H-3-K-8-T-90,BERT-H-3-K-8-T-95
0,0.8592,0.8589,0.8581,0.8582,0.8573,0.8576,0.8587,0.8582,0.8577,0.858,0.8577,0.857


In [ ]:
df_rouge_h1 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_rouge_h_1_k_3_tresh_90.csv")
df_rouge_h2 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_rouge_h_1_k_3_tresh_95.csv")
df_rouge_h3 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_rouge_h_1_k_5_tresh_90.csv")
df_rouge_h4 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_rouge_h_1_k_5_tresh_95.csv")
df_rouge_h5 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_rouge_h_1_k_8_tresh_90.csv")
df_rouge_h6 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_rouge_h_1_k_8_tresh_95.csv")
df_rouge_h7 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_rouge_h_3_k_3_tresh_90.csv")
df_rouge_h8 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_rouge_h_3_k_3_tresh_95.csv")
df_rouge_h9 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_rouge_h_3_k_5_tresh_90.csv")
df_rouge_h10 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_rouge_h_3_k_5_tresh_95.csv")
df_rouge_h11 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_rouge_h_3_k_8_tresh_90.csv")
df_rouge_h12 = pd.read_csv("../resultados/busqueda_hiperparametros/evaluacion_dslr_con_rouge_h_3_k_8_tresh_95.csv")
df_rouge_h1.head()

,Unnamed: 0,chunk,question,answer,ans_temp_0.1_k_3_tresh_90,ans_temp_0.1_k_3_tresh_95,ans_temp_0.1_k_5_tresh_90,ans_temp_0.1_k_5_tresh_95,ans_temp_0.1_k_8_tresh_90,ans_temp_0.1_k_8_tresh_95,ans_temp_0.3_k_3_tresh_90,ans_temp_0.3_k_3_tresh_95,ans_temp_0.3_k_5_tresh_90,ans_temp_0.3_k_5_tresh_95,ans_temp_0.3_k_8_tresh_90,ans_temp_0.3_k_8_tresh_95,ROUGE-1,ROUGE-2,ROUGE-L
0,0,seawater seawater tds mgl considered standard ...,What types of water are classified based on To...,Water is classified into categories like seawa...,"Based on the provided context, water is classi...","Based on the provided context, water can be cl...","Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ...",Water is classified based on Total Dissolved S...,"Based on the provided context, water can be cl...","Based on Total Dissolved Solids (TDS) levels, ...",Water can be classified based on Total Dissolv...,"Based on Total Dissolved Solids (TDS) levels, ...","Based on Total Dissolved Solids (TDS) levels, ...",0.176166,0.073298,0.134715
1,1,recovery limit salinity andor boron concentrat...,Why is it important to limit product recovery ...,Limiting product recovery is important to ensu...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,Limiting product recovery in RO (Reverse Osmos...,0.155642,0.031373,0.101167
2,2,design boynton beach fl membrane softening wat...,How is the maximum recovery value determined f...,The maximum recovery value is determined by co...,The provided context does not explicitly state...,The provided context does not explicitly state...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...,The provided context does not explicitly state...,The provided context does not explicitly state...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...,The maximum recovery value for membrane soften...,0.244186,0.094118,0.174419
3,3,range rather absolute value temperature variat...,Why is average temperature used for performanc...,Average temperature is used because membrane p...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,The average temperature is used for performanc...,0.260870,0.073529,0.217391
4,4,risk scaling due water scarcity environmental ...,Why must scaling substances be removed from tr...,"Even after secondary treatment, wastewater can...",Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be removed from treate...,Scaling substances must be rem

In [23]:
mean_rouge_1=[]
mean_rouge_2=[]
mean_rouge_3=[]
mean_rouge_4=[]
mean_rouge_5=[]
mean_rouge_6=[]
mean_rouge_7=[]
mean_rouge_8=[]
mean_rouge_9=[]
mean_rouge_10=[]
mean_rouge_11=[]
mean_rouge_12=[]

mean_rouge_1.append(df_rouge_h1["ROUGE-1"].mean().round(4))
mean_rouge_2.append(df_rouge_h2["ROUGE-1"].mean().round(4))
mean_rouge_3.append(df_rouge_h3["ROUGE-1"].mean().round(4))
mean_rouge_4.append(df_rouge_h4["ROUGE-1"].mean().round(4))
mean_rouge_5.append(df_rouge_h5["ROUGE-1"].mean().round(4))
mean_rouge_6.append(df_rouge_h6["ROUGE-1"].mean().round(4))
mean_rouge_7.append(df_rouge_h7["ROUGE-1"].mean().round(4))
mean_rouge_8.append(df_rouge_h8["ROUGE-1"].mean().round(4))
mean_rouge_9.append(df_rouge_h9["ROUGE-1"].mean().round(4))
mean_rouge_10.append(df_rouge_h10["ROUGE-1"].mean().round(4))
mean_rouge_11.append(df_rouge_h11["ROUGE-1"].mean().round(4))
mean_rouge_12.append(df_rouge_h12["ROUGE-1"].mean().round(4))

df_mean = pd.DataFrame({

    "rouge-H-1-K-3-T-90": mean_rouge_1,
    "rouge-H-1-K-3-T-95": mean_rouge_2,
    "rouge-H-1-K-5-T-90": mean_rouge_3,
    "rouge-H-1-K-5-T-95": mean_rouge_4,
    "rouge-H-1-K-8-T-90": mean_rouge_5,
    "rouge-H-1-K-8-T-95": mean_rouge_6,
    "rouge-H-3-K-3-T-90": mean_rouge_7,
    "rouge-H-3-K-3-T-95": mean_rouge_8,
    "rouge-H-3-K-5-T-90": mean_rouge_9,
    "rouge-H-3-K-5-T-95": mean_rouge_10,
    "rouge-H-3-K-8-T-90": mean_rouge_11,
    "rouge-H-3-K-8-T-95": mean_rouge_12,
})

In [24]:
df_mean

,rouge-H-1-K-3-T-90,rouge-H-1-K-3-T-95,rouge-H-1-K-5-T-90,rouge-H-1-K-5-T-95,rouge-H-1-K-8-T-90,rouge-H-1-K-8-T-95,rouge-H-3-K-3-T-90,rouge-H-3-K-3-T-95,rouge-H-3-K-5-T-90,rouge-H-3-K-5-T-95,rouge-H-3-K-8-T-90,rouge-H-3-K-8-T-95
0,0.2358,0.2368,0.2289,0.2298,0.2232,0.2267,0.2359,0.2341,0.2292,0.2316,0.2232,0.2282
